# Lab 6.1 — Data Quality Assessment of Chicago 311

*Chapter 6 — Data for AI: Collection, Quality, and Governance · 45 minutes
· JupyterLab with pandas — no API key needed*

A programme office wants to use 311 data to target service improvements.
You have been asked whether the data supports that. Answer with numbers.

Cells marked `# YOUR TURN` ask you to change a simple value and re-run.
No API key is needed.

## Objectives

By the end of this lab, you will:

- Assess a real dataset against all six quality dimensions.
- Quantify each defect rather than describing it.
- Produce a go / no-go recommendation with evidence.

## Setup

**Datasets and files** (all under `labs/data/`):

- `chicago_311.csv` — 4,000 recent City of Chicago 311 service requests
  (real, public; Socrata extract documented in `data/MANIFEST.json`).
  Genuinely messy operational data: blank fields, inconsistent casing and
  duplicate rows are present **on purpose** — they are the subject of the
  lab.

**Tools:** pandas only. No OpenAI key is required for any step.

**Data rule:** this is genuinely public data, downloaded for offline
classroom use. Treat it with the same care you would any operational
extract: address fields in it belong to real residents.

**How this notebook works:** every step is one provided cell — run it with
Shift+Enter and read what it prints. Cells marked `# YOUR TURN` ask you to
change a simple value (like the blank-rate limit that flags a column) and
re-run the cell. Everything runs as shipped, so you can never get stuck.

In [1]:
# ▶ Setup — run this cell first (click it, then Shift+Enter).
# It loads the course helper functions used by every step below.
from lab_helpers import *

## Steps

1. Load the file. (3 min)
2. **Completeness:** null rate per column; flag anything above 20%. (6 min)
3. **Uniqueness:** duplicate `sr_number` count; inspect a duplicate pair.
   (6 min)
4. **Consistency:** case and whitespace variants in categorical columns.
   (6 min)
5. **Validity:** dates outside a plausible range; closed-before-created
   rows. (6 min)
6. **Timeliness:** distribution of `created_date`; how current is it?
   (5 min)
7. **Accuracy:** pick one column and say honestly how you would even test
   it. (5 min)
8. Build a one-table scorecard: dimension, metric, value, pass/fail.
   (5 min)
9. Write a go / no-go recommendation with the two numbers that drive it.
   (3 min)

### Step 1 — Load (3 min, provided)

`data/chicago_311.csv` is a genuine operational extract — 4,000 recent
service requests, warts included on purpose. Run this cell to load it.

In [2]:
df = load_311_data()

shape: (4000, 39)
columns: ['sr_number', 'sr_type', 'sr_short_code', 'created_department', 'owner_department', 'status', 'origin', 'created_date', 'last_modified_date', 'closed_date', 'street_address', 'city', 'state', 'zip_code', 'street_number', 'street_direction', 'street_name', 'street_type', 'duplicate', 'legacy_record', 'legacy_sr_number', 'parent_sr_number', 'community_area', 'ward', 'electrical_district', 'electricity_grid', 'police_sector', 'police_district', 'police_beat', 'precinct', 'sanitation_division_days', 'created_hour', 'created_day_of_week', 'created_month', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 'location']


### Step 2 — Completeness (6 min)

Run this cell for the blank rate per column; it flags anything above 20%.
Watch for columns that are blank for a *reason* (a `closed_date` on an open
request is not a defect — the cell checks that too). Then try a stricter or
looser limit.

In [3]:
NULL_LIMIT = 20   # ← YOUR TURN: the % of blanks that flags a column — try 10 or 50 and re-run this cell
completeness = completeness_report(df, NULL_LIMIT)

legacy_sr_number            100.0
sanitation_division_days    100.0
parent_sr_number             94.5
created_department           69.2
electrical_district          39.5
electricity_grid             39.5
closed_date                  34.8
zip_code                     15.7
city                         14.3
state                        14.3
precinct                      3.3
ward                          3.3
police_beat                   3.3
police_district               3.3
police_sector                 3.3
community_area                3.3
longitude                     3.1
location                      3.1
y_coordinate                  3.1
latitude                      3.1
x_coordinate                  3.1
street_type                   1.0
street_number                 0.1
street_address                0.1
street_direction              0.1
street_name                   0.1

columns above 20% null: 7
closed_date null == open requests? 1390 == 1390


### Step 3 — Uniqueness (6 min, provided)

Run this cell for the duplicate `sr_number` count and a duplicate pair to
inspect. Then look at the city's own `duplicate` flag — do the two stories
agree?

In [4]:
uniqueness = uniqueness_report(df)

duplicated sr_number values: 0
rows flagged duplicate=True by the city: 220
same type + address + timestamp (likely double submissions): 19
    sr_number                  sr_type   street_address            created_date
SR26-01528655 Aircraft Noise Complaint 10510 W ZEMKE RD 2026-07-31T19:21:37.000
SR26-01528654 Aircraft Noise Complaint 10510 W ZEMKE RD 2026-07-31T19:21:37.000
SR26-01529028 Aircraft Noise Complaint 10510 W ZEMKE RD 2026-07-31T20:28:05.000
SR26-01529027 Aircraft Noise Complaint 10510 W ZEMKE RD 2026-07-31T20:28:05.000


### Step 4 — Consistency (6 min, provided)

Run this cell for the case and whitespace variants in the categorical
columns — `city`, `state` — and the way `zip_code` is stored: a ZIP is an
identifier, not a quantity.

In [5]:
consistency_report(df)

city: {'Chicago': 3428, nan: 571, 'CHICAGO': 1}
state: {'Illinois': 3426, nan: 571, 'IL': 3}
zip_dtype: float64
zip_sample: [60666.0, 60614.0, 60618.0]


### Step 5 — Validity (6 min, provided)

Run this cell for dates outside a plausible range and requests closed
before they were created. A zero here is a *pass* — record it as one, with
the check that proved it.

In [6]:
validity = validity_report(df)

{'created_unparseable': 0, 'closed_before_created': 0, 'range': '2026-07-31 19:03:57 -> 2026-08-01 14:14:53'}


### Step 6 — Timeliness (5 min, provided)

Run this cell for the `created_date` coverage: how current is the extract,
and how wide is its window? Currency and coverage are different claims —
assess both.

In [7]:
timeliness = timeliness_report(df)

{'newest': '2026-08-01 14:14:53', 'oldest': '2026-07-31 19:03:57', 'calendar_days': 2, 'window_hours': 19.2}


### Step 7 — Accuracy (5 min)

Pick one column and say honestly how you would *even test* its accuracy.
This is the dimension the data cannot grade itself on — write your answer
in the markdown cell below.

#### Your answer (write here)

*Column chosen, and the test you would run (against what ground truth,
obtained how, at what cost):*

-

### Step 8 — The scorecard (5 min, provided)

Run this cell for the one-table scorecard: dimension, metric, value,
pass/fail, filled from the numbers above. The thresholds baked in are a
defensible starting point — if you defend different ones, say so in your
recommendation.

In [8]:
scorecard = quality_scorecard(df, completeness, uniqueness, validity, timeliness)

   dimension                               metric                                      value passes
Completeness                    columns >20% null                                    7 of 39  False
  Uniqueness                  duplicate sr_number                                          0   True
  Uniqueness              city-flagged duplicates                                        220  False
 Consistency city/state case variants + zip dtype Chicago/CHICAGO, Illinois/IL, zip as float  False
    Validity                closed-before-created                                          0   True
  Timeliness                       window covered                              19.2h, 2 days  False
    Accuracy            testable from data alone?                                         no   None


### Step 9 — Go / no-go recommendation (3 min, write here)

*One paragraph: can the programme office use this file to target service
improvements? Name the two numbers that drive your answer.*

-

## Deliverable
1. The six dimension measurements.
2. The scorecard table.
3. Your go / no-go recommendation with its two driving numbers.

## Reflection
1. Which dimension could you *not* assess from the data alone?
2. What is the single cheapest fix at the point of collection?
3. Would you sign the recommendation you just wrote?

## Debrief (instructor-led)

- **Compare verdicts.** Who said go, who said no-go? Put the two driving
  numbers side by side — did the same number drive opposite verdicts?
- **Thresholds are policy.** The 20% null threshold: who set it, and is it
  the right one for targeting service improvements?
- **Bridge to Lab 6.2.** Which defect class from your scorecard is the
  best candidate for GenAI-assisted cleaning — and which is not?

## Troubleshooting

- **A red error mentioning `chicago_311.csv` or "No such file".** The
  course data pack is missing or incomplete — tell your instructor; run the
  Day-0 healthcheck to confirm.
- **A warning about mixed column types on load.** Expected on this file
  (mixed types in operational columns) — the provided load already handles
  it.
- **A dimension shows zero defects.** That is a pass, not a bug. Record it
  in the scorecard with the check that proved it (Step 5 shows how).
- **`zip_code` prints with a decimal point (60618.0).** That *is* the
  consistency defect Step 4 asks you to find, not a display error.
- **Your numbers differ from a neighbour's.** They should not — every
  number is computed from the same file. Re-run the notebook in order
  (Kernel → Restart & Run All).

---
*Curious about the Python behind these steps? The full code-forward version
of this lab lives in the `For_Python_Programmers/` folder.*